# Figure 2: GTSM output locations and GESLA-3 stations

GTSM coastal output points and the GESLA-3 tide-gauge stations used for validation, with the IPCC AR6 regions.

From Özkan et al. (2025), *Sensitivity of global storm surge modelling to sea surface drag*, Ocean Dynamics, https://doi.org/10.1007/s10236-025-01713-3

The processed input files are not included in this repository; this notebook shows how the figure was made. Experiment names: in the code, `cc` = FIX, `cv` = NST, `ntr`/`uv10n` = CAD and `rho`/`uv10rho` = IFS (see the main README).

In [ ]:
# Folder with the processed input files (not included in this repository)
DATA_DIR = "path/to/paper1_data/"

In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import proplot as pplt
import regionmask
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
gtsm_points_dir = DATA_DIR + "dflow/gtsm4.1_runs/2022.04/"
g3_points_dir = DATA_DIR + "data/4-Processed_for_Visualization/2_global_statistics/OBS_points/"

In [ ]:
# Read gtsm xyn notebook file
gtsm_points_file = gtsm_points_dir + "selected_output_new_unique.xyn"
gtsm_points_df = pd.read_csv(gtsm_points_file, delim_whitespace=True, header=None, names=['longitude', 'latitude', 'station_name'])
#gtsm_points_df = gtsm_points_df.drop_duplicates()
gtsm_coast_points = gtsm_points_df[gtsm_points_df['station_name'].notna() & gtsm_points_df['station_name'].str.contains('_coast_')]
# Then dop the station_name column
gtsm_coast_points = gtsm_coast_points.drop(columns=['station_name'])

In [ ]:
g3_points_file = g3_points_dir + "0-GESLA3.nc"
g3_points_ds = xr.open_dataset(g3_points_file)

In [ ]:
g3_longitudes = g3_points_ds['longitude'].values
g3_latitudes = g3_points_ds['latitude'].values
gtsm_longitudes = gtsm_coast_points['longitude'].values
gtsm_latitudes = gtsm_coast_points['latitude'].values

In [ ]:
#mask_3d_g3 = regionmask.defined_regions.ar6.land.mask_3D(g3_points_ds, lon_name="longitude", lat_name="latitude")
mask_3d_g3 = regionmask.defined_regions.ar6.land.mask_3D(g3_points_ds['longitude'], g3_points_ds['latitude'])
mask_3d_g3.set_index(regions=["region", "abbrevs", "names"])
masked_g3 =  mask_3d_g3.where(mask_3d_g3 == True, drop=True)

In [ ]:
gtsm_coast_points_ds = xr.Dataset({'longitude': (['points'], gtsm_longitudes),
                                    'latitude': (['points'], gtsm_latitudes)})

mask_3d_gtsm = regionmask.defined_regions.ar6.land.mask_3D(
    gtsm_coast_points_ds['longitude'], 
    gtsm_coast_points_ds['latitude'])

mask_3d_gtsm = mask_3d_gtsm.assign_coords(
    longitude=gtsm_coast_points_ds['longitude'],
    latitude=gtsm_coast_points_ds['latitude'])

mask_3d_gtsm = mask_3d_gtsm.set_index(regions=["region", "abbrevs", "names"])
masked_gtsm = mask_3d_gtsm.where(mask_3d_gtsm == True, drop=True)

In [ ]:
pplt.rc['reso'] = 'xx-hi'  # 'lo', 'med', 'hi', 'x-hi', 'xx-hi'
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={'projection': ccrs.Robinson()})

# Add the scatter plots for the model and observation points
scatter_gtsm = ax.scatter(masked_gtsm.longitude, masked_gtsm.latitude, c='sapphire', s=8, 
                          edgecolor='none', transform=ccrs.PlateCarree(), zorder=2, label='GTSM Results')
scatter_g3 = ax.scatter(masked_g3.longitude, masked_g3.latitude, c="cherry", s=9, #marker='*',
                        edgecolor='none', transform=ccrs.PlateCarree(), zorder=2, label='GESLA-3 Stations')

# Plot the AR6 land regions using Cartopy GeoAxes
map_region = regionmask.defined_regions.ar6.land
text_kws = dict(fontsize=12, color='mahogany', ha='center', va='center', zorder=3, bbox=dict(color="none"),
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])
map_region.plot(ax=ax, text_kws=text_kws, label='abbrev')
#map_region.plot(ax=ax, line_kws=dict(color='black', linewidth=1.0), add_label=False)

# Add a legend to distinguish between the model results and observation points
legend = ax.legend(
    loc='lower center',
    fontsize=13,
    ncol=2,
    bbox_to_anchor=(0.6, 0.001),
    handletextpad=0.00001,  # Reduce space between the marker and the text
    columnspacing=0.3  # Reduce space between columns
)

# Increase the size of the dots in the legend
for handle in legend.legendHandles:
    handle.set_sizes([60])

# Format gridlines and labels with improved readability
ax.set_global()
ax.coastlines(linewidth=0.2, color='grey')
ax.add_feature(cfeature.LAND, color='lightgrey')
ax.add_feature(cfeature.OCEAN, color='lightblue')

# Gridlines with labels at the bottom (longitude) and left (latitude) only
gridlines = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--',
                         xlocs=np.arange(-180, 181, 20), ylocs=np.arange(-90, 91, 10))

# Set label styles
gridlines.xlabel_style = {'size': 9.8, 'color': 'black'}
gridlines.ylabel_style = {'size': 9.8, 'color': 'black'}
gridlines.xlabels_top = False
gridlines.ylabels_right = False

ax.set_extent([-180, 180, -61, 82], crs=ccrs.PlateCarree())

# Adjust layout to remove unnecessary whitespace
plt.tight_layout()

# Save the plot as PDF and PNG, cropping the bounding box
# Define the target directory
save_dir = 'final_plots'
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save the plot as PDF and PNG in the specified directory
fig.savefig(os.path.join(save_dir, 'global_gtsm_gesla3_map.pdf'), format='pdf', dpi=600, bbox_inches='tight')
fig.savefig(os.path.join(save_dir, 'global_gtsm_gesla3_map.png'), dpi=600, bbox_inches='tight')

# Show the plot
plt.show()